# Teste de diferentes classificadores para Isquemia

# OCPC Classifier

In [ ]:
import pandas as pd
import numpy as np
import joblib 
import os
import cv2
import time
from sklearn.model_selection import KFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from ocpc_py import MultiClassPC
from codecarbon import EmissionsTracker

# Configurações
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Monta dataset
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))

df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carrega e normaliza imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Validação Cruzada - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    pca = PCA(n_components=50)
    X_tr_pca = pca.fit_transform(X_tr)
    X_val_pca = pca.transform(X_val)

    clf = MultiClassPC()
    clf.fit(X_tr_pca, y_tr)
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Resultados finais do K-Fold
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# TREINAMENTO FINAL PARA PRODUÇÃO
print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

pca_final = PCA(n_components=50)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

clf_final = MultiClassPC()
clf_final.fit(X_train_pca, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante todo o experimento: {emissions:.6f} kg CO₂eq")

# Avaliação no conjunto de teste
y_pred_test = clf_final.predict(X_test_pca)
y_proba_test = clf_final.predict_proba(X_test_pca)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
pca_path = os.path.join(output_dir, "OCPC_pca_ischaemia.pkl")
clf_path = os.path.join(output_dir, "OCPC_modelo_ischaemia.pkl")
joblib.dump(pca_final, pca_path)
joblib.dump(clf_final, clf_path)

# Random Forest

## Random Forest com PCA

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from sklearn.model_selection import KFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from codecarbon import EmissionsTracker 
import joblib

# Configurações
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Monta dataset
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))

df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carrega e normaliza imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Validação Cruzada - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    pca = PCA(n_components=50)
    X_tr_pca = pca.fit_transform(X_tr)
    X_val_pca = pca.transform(X_val)

    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_tr_pca, y_tr)
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Resultados finais do K-Fold
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# TREINAMENTO FINAL PARA PRODUÇÃO
print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

pca_final = PCA(n_components=50)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

clf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_final.fit(X_train_pca, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante o treinamento final: {emissions:.6f} kg CO₂eq")

# Avaliação no conjunto de teste
y_pred_test = clf_final.predict(X_test_pca)
y_proba_test = clf_final.predict_proba(X_test_pca)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

# Medindo tempo de classificação (em segundos)
tempos_inferencia = []
for x in X_test_pca:
    inicio = time.time()
    _ = clf_final.predict([x])
    fim = time.time()
    tempos_inferencia.append(fim - inicio)  

tempos_inferencia = np.array(tempos_inferencia)
print("\n[TEMPO DE CLASSIFICAÇÃO]")
print(f"Tempo médio por imagem: {np.mean(tempos_inferencia):.4f} s ± {np.std(tempos_inferencia):.4f} s")

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
pca_path = os.path.join(output_dir, "random_forest_PCA_ischaemia.pkl")
clf_path = os.path.join(output_dir, "random_forest_modelo_PCA_ischaemia.pkl")
joblib.dump(pca_final, pca_path)
joblib.dump(clf_final, clf_path)


## Random Forest sem PCA

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
import time
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import joblib
from codecarbon import EmissionsTracker 

# Caminho base e configuração
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Montar dataset (imagem, rótulo)
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))
df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carregar e normalizar imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)

# Separar treino/teste (20% é o holdout final)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# VALIDAÇÃO CRUZADA (K-Fold no treino)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Cross Validation - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Exibir métricas médias
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

clf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_final.fit(X_train, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante o treinamento final: {emissions:.6f} kg CO₂eq")

# Avaliação em teste
y_pred_test = clf_final.predict(X_test)
y_proba_test = clf_final.predict_proba(X_test)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

# Medindo tempo de classificação (em segundos)
tempos_inferencia = []
for x in X_test:
    inicio = time.time()
    _ = clf_final.predict([x])
    fim = time.time()
    tempos_inferencia.append(fim - inicio)  

tempos_inferencia = np.array(tempos_inferencia)
print("\n[TEMPO DE CLASSIFICAÇÃO]")
print(f"Tempo médio por imagem: {np.mean(tempos_inferencia):.4f} s ± {np.std(tempos_inferencia):.4f} s")

#Salvando o modelo final
output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
clf_path = os.path.join(output_dir, "random_forest_ischaemia.pkl")
joblib.dump(clf_final, clf_path)


# XGBOOST

## XGBOOST com PCA

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier
from codecarbon import EmissionsTracker 

base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Monta dataset
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))

df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carrega e normaliza imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"[AVISO] Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Validação Cruzada - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    pca = PCA(n_components=50)
    X_tr_pca = pca.fit_transform(X_tr)
    X_val_pca = pca.transform(X_val)

    clf = XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        scale_pos_weight=1, random_state=42, n_jobs=-1
    )
    clf.fit(X_tr_pca, y_tr)
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Resultados finais do K-Fold
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# TREINAMENTO FINAL PARA PRODUÇÃO
print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

pca_final = PCA(n_components=50)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

clf_final = XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        scale_pos_weight=1, random_state=42, n_jobs=-1
    )
clf_final.fit(X_train_pca, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante o treinamento final: {emissions:.6f} kg CO₂eq")

# Avaliação no conjunto de teste
y_pred_test = clf_final.predict(X_test_pca)
y_proba_test = clf_final.predict_proba(X_test_pca)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

# Medindo tempo de classificação (em segundos)
tempos_inferencia = []
for x in X_test_pca:
    inicio = time.time()
    _ = clf_final.predict([x])
    fim = time.time()
    tempos_inferencia.append(fim - inicio)  

tempos_inferencia = np.array(tempos_inferencia)
print("\n[TEMPO DE CLASSIFICAÇÃO]")
print(f"Tempo médio por imagem: {np.mean(tempos_inferencia):.4f} s ± {np.std(tempos_inferencia):.4f} s")

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
pca_path = os.path.join(output_dir, "XGB_PCA_ischaemia.pkl")
clf_path = os.path.join(output_dir, "XGB_modelo_PCA_ischaemia.pkl")
joblib.dump(pca_final, pca_path)
joblib.dump(clf_final, clf_path)


## XGBOOST sem PCA

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier
from codecarbon import EmissionsTracker 

# Caminho base e configurações
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Montar dataframe com caminhos e rótulos
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        dataset.append((os.path.join(pasta, imagem), label))
df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Função para carregar imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"[AVISO] Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)

# Separar treino/teste (20% é o holdout final)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# VALIDAÇÃO CRUZADA (K-Fold no treino)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Cross Validation - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    clf = XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        scale_pos_weight=1, random_state=42, n_jobs=-1
    )
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Exibir métricas médias
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

clf_final = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    scale_pos_weight=1, random_state=42, n_jobs=-1
)
clf_final.fit(X_train, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante o treinamento final: {emissions:.6f} kg CO₂eq")

# Avaliação em teste
y_pred_test = clf_final.predict(X_test)
y_proba_test = clf_final.predict_proba(X_test)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

# Gerar e plotar matriz de confusão
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusão - Xgboost (Isquemia)")
plt.xlabel("Predito")
plt.ylabel("Verdadeiro")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "confusion_matrix_ischaemia.png"))  # Salva imagem
plt.show()

# Medindo tempo de classificação (em segundos)
import time
tempos_inferencia = []
for x in X_test:
    inicio = time.time()
    _ = clf_final.predict([x])
    fim = time.time()
    tempos_inferencia.append(fim - inicio)  

tempos_inferencia = np.array(tempos_inferencia)
print("\n[TEMPO DE CLASSIFICAÇÃO]")
print(f"Tempo médio por imagem: {np.mean(tempos_inferencia):.4f} s ± {np.std(tempos_inferencia):.4f} s")

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
clf_path = os.path.join(output_dir, "XGB_ischaemia.pkl")
joblib.dump(clf_final, clf_path)


# MLP

## MLP com PCA

In [ ]:
# ISCHAEMIA COM PCA E MLPClassifier
import pandas as pd
import numpy as np
import os
import cv2
import joblib
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.neural_network import MLPClassifier
from codecarbon import EmissionsTracker 

# Configurações
base_path = "../dataset/ischaemia/"
image_size = (64, 64)

# Monta dataset
dataset = []
for classe, label in zip(["Aug-Positive", "Aug-Negative"], [1, 0]):
    pasta = os.path.join(base_path, classe)
    for imagem in os.listdir(pasta):
        caminho_imagem = os.path.join(pasta, imagem)
        dataset.append((caminho_imagem, label))

df = pd.DataFrame(dataset, columns=["imagem", "label"])

# Carrega e normaliza imagens
def load_images(df, image_size):
    imagens, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row["imagem"])
        if img is not None:
            img = cv2.resize(img, image_size)
            img = img.astype("float32") / 255.0
            imagens.append(img.flatten())
            labels.append(row["label"])
        else:
            print(f"Imagem não carregada: {row['imagem']}")
    return np.array(imagens), np.array(labels)

X_all, y_all = load_images(df, image_size)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs, precs, recs, f1s, aucs = [], [], [], [], []

print("\n[Validação Cruzada - Treinamento]")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    pca = PCA(n_components=50)
    X_tr_pca = pca.fit_transform(X_tr)
    X_val_pca = pca.transform(X_val)

    clf = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=42
    )
    clf.fit(X_tr_pca, y_tr)
    y_pred = clf.predict(X_val_pca)
    y_proba = clf.predict_proba(X_val_pca)[:, 1]

    accs.append(accuracy_score(y_val, y_pred))
    precs.append(precision_score(y_val, y_pred))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    aucs.append(roc_auc_score(y_val, y_proba))

    print(f"\n[Fold {fold}]")
    print(f"Acurácia: {accs[-1]:.4f} | Precisão: {precs[-1]:.4f} | Recall: {recs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUC: {aucs[-1]:.4f}")

# Resultados finais do K-Fold
print("\n[Métricas Médias - Cross-Validation]")
print(f"Acurácia: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Precisão: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall:   {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"AUC:      {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

# TREINAMENTO FINAL PARA PRODUÇÃO
print("\n[Treinamento Final e Avaliação em Teste]")
tracker = EmissionsTracker(log_level="ERROR")  # Desabilitar logs detalhados
tracker.start()

pca_final = PCA(n_components=50)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

clf_final = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    random_state=42
)
clf_final.fit(X_train_pca, y_train)

emissions = tracker.stop()
print(f"\n[Pegada de Carbono]")
print(f"Emissões estimadas durante o treinamento final: {emissions:.6f} kg CO₂eq")

# Avaliação no conjunto de teste
y_pred_test = clf_final.predict(X_test_pca)
y_proba_test = clf_final.predict_proba(X_test_pca)[:, 1]

print("\n[Desempenho no Conjunto de Teste]")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precisão: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:   {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba_test):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_test))

# Medindo tempo de classificação (em segundos)
tempos_inferencia = []
import time
for x in X_test_pca:
    inicio = time.time()
    _ = clf_final.predict([x])
    fim = time.time()
    tempos_inferencia.append(fim - inicio)  

tempos_inferencia = np.array(tempos_inferencia)
print("\n[TEMPO DE CLASSIFICAÇÃO]")
print(f"Tempo médio por imagem: {np.mean(tempos_inferencia):.4f} s ± {np.std(tempos_inferencia):.4f} s")

output_dir = "../models/ischaemia/"
os.makedirs(output_dir, exist_ok=True)
pca_path = os.path.join(output_dir, "MLP_PCA_ischaemia.pkl")
clf_path = os.path.join(output_dir, "MLP_modelo_PCA_ischaemia.pkl")
joblib.dump(pca_final, pca_path)
joblib.dump(clf_final, clf_path)

